# GeoVision: 660 × 600, até três horas

Ative **Ambiente de execução → Alterar tipo → GPU** e execute na ordem.
Este notebook contém os módulos necessários. Não apaga dados do Drive.
Treina uma cabeça sobre MobileNetV2 congelada, com teto alto de épocas e parada por tempo.
Reserva 20 minutos das 3h para finalizar. O limite é cooperativo; operações bloqueadas podem excedê-lo.
Instalação e montagem do Drive ficam fora do relógio.

Use `visual` para fotos rotuladas ou `multimodal` para casos reais com três fotos,
triagem e revisor. Não preencha triagem fictícia. As células antigas de clone,
extração de ZIP e `split-folders` não fazem parte deste notebook.

No modo visual, o resultado é um `.keras` 600 × 660; a API deste projeto já o
aceita quando `CAMINHO_MODELO` aponta para esse arquivo. O modo multimodal ainda
exige uma integração própria, pois recebe quatro entradas.


In [ ]:
%pip -q install scikit-learn pillow
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf
print('TensorFlow:', tf.__version__)
assert tf.config.list_physical_devices('GPU'), 'Selecione GPU no Colab e reconecte.'


In [ ]:
from pathlib import Path
import sys
FONTES = {'contrato_modelo.py': '"""Contrato compartilhado pelo Colab e API. Não contém pesos de engenharia."""\nfrom __future__ import annotations\n\nimport io\nimport math\nimport numpy as np\nfrom PIL import Image, ImageOps\n\nCLASSES = ["baixo", "critico", "medio", "sem_risco"]\nALTURA, LARGURA = 600, 660\nPERSPECTIVAS = ["geral", "detalhe", "escala"]\n# A posição de cada categoria é parte do artefato, inclusive \'desconhecido\'.\nVOCABULARIO = {\n    "tipo_anomalia": ["desconhecido", "rachadura", "inclinacao_muro", "infiltracao", "desplacamento", "armadura_exposta", "afundamento_recalque", "outro"],\n    "evolucao": ["desconhecido", "estavel", "aumentando", "rapido"],\n    "local_anomalia": ["desconhecido", "parede", "viga_pilar", "laje_piso", "muro_arrimo", "solo_talude", "outro"],\n    "ruido_percebido": ["desconhecido", "nenhum", "estalos", "vibracao_ao_pisar", "outro"],\n    "gravidade_percebida": ["desconhecido", "baixo", "medio", "alto"],\n    "tempo_surgimento": ["desconhecido", "recente", "semanas", "meses"],\n}\n\n# Escalas numéricas para condicionamento da rede, NÃO limites de segurança.\n# Só preencher com medições verificadas e com unidades indicadas.\nMEDICOES = {"abertura_mm": 10.0, "evolucao_mm_dia": 1.0,\n            "desaprumo_mm_m": 10.0, "distorcao_angular": 0.01}\n\n\ndef codificar_triagem(dados: dict, vocabulario: dict = VOCABULARIO) -> np.ndarray:\n    valores = []\n    for campo, categorias in vocabulario.items():\n        valor = dados.get(campo)\n        valor = valor if valor in categorias else "desconhecido"\n        valores.extend(float(categoria == valor) for categoria in categorias)\n    for campo, escala in MEDICOES.items():\n        bruto = dados.get(campo)\n        ausente = bruto is None or (isinstance(bruto, str) and not bruto.strip())\n        valor = 0.0 if ausente else float(bruto)\n        if isinstance(bruto, bool) or not math.isfinite(valor) or (campo != "evolucao_mm_dia" and valor < 0):\n            raise ValueError(f"Medição inválida: {campo}")\n        valores.extend([valor / escala, float(not ausente)])\n    return np.asarray(valores, dtype=np.float32)\n\n\ndef preparar_imagem(conteudo: bytes, altura: int = ALTURA, largura: int = LARGURA) -> np.ndarray:\n    """RGB 0..255, EXIF e letterbox idênticos nos dois ambientes.\n\n    Normalização -1..1 fica dentro do novo modelo. Não corta a escala da foto.\n    """\n    with Image.open(io.BytesIO(conteudo)) as original:\n        imagem = ImageOps.exif_transpose(original).convert("RGB")\n        imagem = ImageOps.pad(imagem, (largura, altura), method=Image.Resampling.BILINEAR, color=(127, 127, 127))\n        return np.asarray(imagem, dtype=np.float32)\n\n\ndef metadados_modelo(modo: str) -> dict:\n    if modo not in {"visual", "multimodal"}:\n        raise ValueError("Modo deve ser visual ou multimodal")\n    return {\n        "schema_version": 2, "modo": modo, "classes": CLASSES,\n        "altura": ALTURA, "largura": LARGURA,\n        "preprocessamento": "pil_letterbox_rgb_0_255_rescaling_interno",\n        "perspectivas": PERSPECTIVAS,\n        "vocabulario": VOCABULARIO,\n        "medicoes_escalas": MEDICOES,\n        "medicoes_layout": "valor_dividido_pela_escala,indicador_presenca",\n        "homologado": False,\n        "observacao": "Modelo experimental de triagem; softmax não é probabilidade de ruína.",\n    }\n', 'treinar_colab.py': '"""Treinamento 660 x 600: extração congelada + épocas limitadas pelo relógio.\n\nExecute no notebook GeoVision_660x600.ipynb. Não apaga o dataset do Drive.\nO prazo é cooperativo: download/IO/kernel travado não têm limite rígido.\n"""\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport math\nimport shutil\nimport sys\nimport time\nfrom datetime import datetime, timezone\n\nimport numpy as np\n\nif "__file__" in globals():\n    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "ai-service/app/services"))\nfrom contrato_modelo import (\n    ALTURA, LARGURA, CLASSES, PERSPECTIVAS, VOCABULARIO,\n    codificar_triagem, preparar_imagem, metadados_modelo,\n)\n\n\ndef ler_manifesto(raiz, modo, avaliacao_exploratoria=False):\n    """Exige edifícios conhecidos, salvo experimento visual explicitamente habilitado."""\n    nome = "casos.csv" if modo == "multimodal" else "imagens.csv"\n    arquivo = raiz / nome\n    if avaliacao_exploratoria and modo != "visual":\n        raise ValueError("Origem desconhecida só é permitida no modo visual experimental.")\n    if not arquivo.exists():\n        ajuda = "Execute a célula \'Preparar a lista de imagens\', preencha edificio_id no inventário e salve como imagens.csv." if modo == "visual" else "Prepare casos.csv com fotos e triagem de casos reais revisados."\n        raise ValueError(f"Arquivo não encontrado: {arquivo}. {ajuda}")\n    with arquivo.open(encoding="utf-8-sig", newline="") as f:\n        linhas = list(csv.DictReader(f))\n    fotos = PERSPECTIVAS if modo == "multimodal" else ["imagem"]\n    obrigatorias = ["caso_id", "edificio_id", "rotulo", *fotos]\n    if modo == "multimodal":\n        obrigatorias += ["revisor", *VOCABULARIO]\n    if not linhas or any(c not in linhas[0] for c in obrigatorias):\n        raise ValueError(f"CSV vazio ou sem colunas: {obrigatorias}")\n    vistos, hashes, rotulos_hash = set(), {}, {}\n    for row in linhas:\n        if not (row["caso_id"] or "").strip() or row["caso_id"] in vistos or (not avaliacao_exploratoria and not (row["edificio_id"] or "").strip()):\n            raise ValueError("caso_id deve ser único e edificio_id obrigatório.")\n        if avaliacao_exploratoria and (row["edificio_id"] or "").strip():\n            raise ValueError("Modo exploratório é para origem desconhecida. Para IDs conhecidos, desative a opção; não misture origens conhecidas e desconhecidas.")\n        vistos.add(row["caso_id"])\n        if row["rotulo"] not in CLASSES:\n            raise ValueError(f"Rótulo inválido: {row[\'rotulo\']}")\n        if modo == "multimodal" and not row["revisor"].strip():\n            raise ValueError("Casos multimodais exigem identificação do revisor técnico.")\n        if modo == "multimodal":\n            codificar_triagem(row)  # Rejeita medições inválidas antes de consumir GPU.\n        for campo, opcoes in VOCABULARIO.items():\n            if row.get(campo) and row[campo] not in opcoes:\n                raise ValueError(f"Categoria inválida em {campo}: {row[campo]}")\n        hashes_caso = set()\n        for campo in fotos:\n            caminho = (raiz / row[campo]).resolve()\n            if not caminho.is_relative_to(raiz.resolve()) or not caminho.is_file():\n                raise ValueError(f"Imagem ausente ou fora do dataset: {row[campo]}")\n            digest = hashlib.sha256(caminho.read_bytes()).hexdigest()\n            if digest in rotulos_hash and rotulos_hash[digest] != row["rotulo"]:\n                raise ValueError("Foto idêntica com rótulos diferentes. Revise as classes antes de treinar.")\n            rotulos_hash[digest] = row["rotulo"]\n            if avaliacao_exploratoria:\n                row["_grupo_split"] = digest\n            if digest in hashes_caso:\n                raise ValueError(f"Perspectivas repetidas no caso {row[\'caso_id\']}")\n            hashes_caso.add(digest)\n            if digest in hashes and hashes[digest] != row["edificio_id"]:\n                raise ValueError("Foto duplicada atribuída a edifícios distintos. Corrija a origem.")\n            hashes[digest] = row["edificio_id"]\n    if avaliacao_exploratoria:\n        # Uma cópia por conteúdo evita inflar métricas com arquivos repetidos.\n        unicas = {}\n        for row in linhas:\n            unicas.setdefault(row["_grupo_split"], row)\n        print(f"Avaliação EXPLORATÓRIA: {len(linhas)} arquivos, {len(unicas)} conteúdos únicos. Origem dos imóveis desconhecida.")\n        linhas = list(unicas.values())\n    return linhas, fotos\n\n\ndef dividir(linhas):\n    from sklearn.model_selection import StratifiedGroupKFold\n    y = np.array([CLASSES.index(r["rotulo"]) for r in linhas])\n    grupos = np.array([r.get("_grupo_split", r["edificio_id"]) for r in linhas])\n    unidade = "conteúdos únicos" if any("_grupo_split" in r for r in linhas) else "edifícios"\n    for classe in range(len(CLASSES)):\n        if len(set(grupos[y == classe])) < 5:\n            raise ValueError(f"{CLASSES[classe]} precisa aparecer em ao menos 5 {unidade} para separar treino/val/teste; isso é um mínimo operacional, não suficiência estatística.")\n    # Aproximadamente 60/20/20 por edifício ou conteúdo no experimento visual.\n    # Teste nunca escolhe épocas.\n    folds = list(StratifiedGroupKFold(5, shuffle=True, random_state=42).split(y, y, grupos))\n    teste, val = folds[0][1], folds[1][1]\n    treino = np.setdiff1d(np.arange(len(y)), np.concatenate([teste, val]))\n    for indices in (treino, val, teste):\n        if set(y[indices]) != set(range(len(CLASSES))):\n            raise ValueError(f"Split por {unidade} ficou sem alguma classe. Amplie/reorganize o dataset mantendo os grupos.")\n    return y, treino, val, teste\n\n\ndef treinar(raiz, modo="multimodal", horas=3.0, epocas=1_000_000, batch_imagens=8, avaliacao_exploratoria=False):\n    inicio = time.monotonic()\n    prazo = inicio + horas * 3600\n    # Reserva para teste, remontagem, salvamento local e cópia ao Drive.\n    fim_treino = prazo - 20 * 60\n    if not math.isfinite(horas) or not 1 / 3 < horas <= 3 or epocas < 1 or batch_imagens < 1:\n        raise ValueError("Orçamento deve superar 20 min e ser no máximo 3h; épocas e batch positivos.")\n    if modo not in {"visual", "multimodal"}:\n        raise ValueError("Modo inválido")\n    import tensorflow as tf\n    from sklearn.metrics import classification_report, confusion_matrix\n    tf.keras.utils.set_random_seed(42)\n    if not tf.config.list_physical_devices("GPU"):\n        raise RuntimeError("Ative GPU no Colab antes de iniciar; não prometemos 3 horas em CPU.")\n    raiz = Path(raiz)\n    linhas, fotos = ler_manifesto(raiz, modo, avaliacao_exploratoria=avaliacao_exploratoria)\n    y, treino, val, teste = dividir(linhas)\n    sessao = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")\n    local = Path("/content") / f"geovision_{sessao}"\n    local.mkdir(parents=True)\n    destino = raiz / "treinamentos" / sessao\n    destino.mkdir(parents=True)\n    manifesto = [{"caso_id": r["caso_id"], "edificio_id": r["edificio_id"], "grupo_split": r.get("_grupo_split", r["edificio_id"]), "imagem": r.get("imagem"), "split": "treino" if i in treino else "val" if i in val else "teste"} for i, r in enumerate(linhas)]\n    (destino / "split.json").write_text(json.dumps(manifesto, indent=2), encoding="utf-8")\n\n    # Entrada em pixels 0..255; normalização exportada dentro do modelo.\n    entrada = tf.keras.Input((ALTURA, LARGURA, 3), name="imagem")\n    pixels = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(entrada)\n    base = tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(ALTURA, LARGURA, 3), pooling="avg")\n    base.trainable = False\n    extrator = tf.keras.Model(entrada, base(pixels, training=False), name="extrator")\n\n    # Copia só os arquivos do manifesto para disco local; nenhuma remoção no Drive.\n    caminhos = []\n    for row in linhas:\n        lista = []\n        for campo in fotos:\n            origem = raiz / row[campo]\n            alvo = local / "imagens" / row[campo]\n            alvo.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(origem, alvo)\n            lista.append(alvo)\n        caminhos.append(lista)\n        if time.monotonic() >= fim_treino:\n            raise TimeoutError("Cópia consumiu o orçamento; nenhum modelo foi publicado.")\n\n    n, vistas = len(linhas), len(fotos)\n    features = np.zeros((n, vistas * 1280), dtype=np.float32)\n    # Uma passagem pela CNN. Batch pequeno e redução automática se faltar VRAM.\n    itens = [(i, j, p) for i, paths in enumerate(caminhos) for j, p in enumerate(paths)]\n    cursor, lote, tempos = 0, batch_imagens, []\n    while cursor < len(itens):\n        if time.monotonic() >= fim_treino:\n            raise TimeoutError("Extração excedeu orçamento; reduza dataset ou resolução. Não exportado modelo incompleto.")\n        selecao = itens[cursor:cursor + lote]\n        batch = np.stack([preparar_imagem(p.read_bytes()) for _, _, p in selecao])\n        t = time.monotonic()\n        try:\n            valores = extrator(batch, training=False).numpy()\n        except tf.errors.ResourceExhaustedError:\n            if lote == 1:\n                raise RuntimeError("GPU sem memória para 600x660; use resolução menor em treino E API.")\n            lote = max(1, lote // 2)\n            continue\n        tempos.append((time.monotonic() - t) / len(selecao))\n        for (i, j, _), vetor in zip(selecao, valores):\n            features[i, j * 1280:(j + 1) * 1280] = vetor\n        cursor += len(selecao)\n        if cursor == len(selecao) or cursor % (lote * 20) == 0:\n            estimativa = np.median(tempos[-10:]) * (len(itens) - cursor)\n            print(f"Extração {cursor}/{len(itens)}; restante estimado CNN: {estimativa / 60:.1f} min", flush=True)\n    np.save(local / "features.npy", features)\n    entradas_head = [tf.keras.Input((vistas * 1280,), name="features")]\n    x = entradas_head[0]\n    dados = {"features": features}\n    if modo == "multimodal":\n        triagem = np.stack([codificar_triagem(r) for r in linhas])\n        entrada_triagem = tf.keras.Input((triagem.shape[1],), name="triagem")\n        entradas_head.append(entrada_triagem)\n        x = tf.keras.layers.Concatenate()([x, entrada_triagem])\n        dados["triagem"] = triagem\n    x = tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-3))(x)\n    x = tf.keras.layers.Dropout(.4)(x)\n    saida = tf.keras.layers.Dense(len(CLASSES), activation="softmax", dtype="float32")(x)\n    head = tf.keras.Model(entradas_head, saida, name="decisor")\n    head.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])\n    contar = np.bincount(y[treino], minlength=len(CLASSES))\n    pesos = {i: len(treino) / (len(CLASSES) * int(q)) for i, q in enumerate(contar)}\n\n    class Orcamento(tf.keras.callbacks.Callback):\n        def __init__(self):\n            super().__init__()\n            self.tempos = []\n        def on_epoch_begin(self, epoch, logs=None):\n            self.inicio_epoca = time.monotonic()\n            self.interrompida = False\n        def on_train_batch_end(self, batch, logs=None):\n            if time.monotonic() >= fim_treino:\n                self.model.stop_training = True\n                self.interrompida = batch + 1 < self.params["steps"]\n        def on_epoch_end(self, epoch, logs=None):\n            # Não promover um checkpoint de uma época incompleta/não finita.\n            if not self.interrompida and logs and math.isfinite(logs.get("val_loss", float("nan"))):\n                if logs["val_loss"] < self.melhor_loss:\n                    self.model.save_weights(str(melhor))\n                    self.melhor_loss = logs["val_loss"]\n                    self.melhor_epoca = epoch + 1\n            self.tempos.append(time.monotonic() - self.inicio_epoca)\n            restante = fim_treino - time.monotonic()\n            if epoch == 0:\n                print(f"Estimativa para {epocas} épocas da cabeça: {self.tempos[-1] * epocas / 60:.1f} min")\n            if restante < 1.3 * max(self.tempos[-3:]):\n                self.model.stop_training = True\n\n    melhor = local / "melhor.weights.h5"\n    orcamento = Orcamento()\n    orcamento.melhor_loss = float("inf")\n    orcamento.melhor_epoca = None\n    if time.monotonic() >= fim_treino:\n        raise TimeoutError("Sem tempo restante para treinar.")\n    historico = head.fit(\n        {k: v[treino] for k, v in dados.items()}, y[treino],\n        validation_data=({k: v[val] for k, v in dados.items()}, y[val]),\n        epochs=epocas, batch_size=32, class_weight=pesos,\n        callbacks=[orcamento, tf.keras.callbacks.TerminateOnNaN(),\n                   tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=8, factor=.5, min_lr=1e-5)],\n        verbose=2,\n    )\n    if not melhor.exists():\n        raise RuntimeError("Nenhum checkpoint válido; não exportar.")\n    head.load_weights(melhor)\n    previsoes = head.predict({k: v[teste] for k, v in dados.items()}, verbose=0)\n    relatorio = classification_report(y[teste], previsoes.argmax(1), labels=list(range(4)), target_names=CLASSES, output_dict=True, zero_division=0)\n    relatorio["matriz_confusao"] = confusion_matrix(y[teste], previsoes.argmax(1), labels=list(range(4))).tolist()\n    relatorio["epocas_executadas"] = len(historico.epoch)\n    relatorio["melhor_epoca"] = orcamento.melhor_epoca\n    relatorio["orcamento_horas"] = horas\n    relatorio["reserva_finalizacao_minutos"] = 20\n    relatorio["tipo_avaliacao"] = "exploratoria_por_conteudo" if avaliacao_exploratoria else "por_edificio"\n    relatorio["origem_imoveis_conhecida"] = not avaliacao_exploratoria\n    relatorio["segundos_ate_avaliacao"] = time.monotonic() - inicio\n    # Artefato final recebe as fotos, não embeddings pré-computados do cliente.\n    novas = [tf.keras.Input((ALTURA, LARGURA, 3), name=nome) for nome in fotos]\n    vetores = [extrator(foto, training=False) for foto in novas]\n    vetor = tf.keras.layers.Concatenate()(vetores) if len(vetores) > 1 else vetores[0]\n    entrada_final = {"features": vetor}\n    if modo == "multimodal":\n        t = tf.keras.Input((dados["triagem"].shape[1],), name="triagem")\n        novas.append(t)\n        entrada_final["triagem"] = t\n    modelo = tf.keras.Model(novas, head(entrada_final, training=False))\n    artefato = local / "geovision_model_pronto.keras"\n    modelo.save(artefato)\n    meta = metadados_modelo(modo)\n    meta["tipo_avaliacao"] = relatorio["tipo_avaliacao"]\n    if avaliacao_exploratoria:\n        meta["limitacao_avaliacao"] = "Fotos diferentes do mesmo imóvel podem estar em splits distintos; não demonstra generalização para outros imóveis."\n    meta.update({"versao": sessao, "tensorflow": tf.__version__, "sha256": hashlib.sha256(artefato.read_bytes()).hexdigest()})\n    (local / "geovision_model_pronto.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")\n    (local / "avaliacao.json").write_text(json.dumps(relatorio, ensure_ascii=False, indent=2), encoding="utf-8")\n    (local / "historico.json").write_text(json.dumps(historico.history, indent=2), encoding="utf-8")\n    # Confere serialização e equivalência entre treino em features e inferência.\n    recarregado = tf.keras.models.load_model(artefato, compile=False)\n    idx = int(teste[0])\n    amostra = [np.expand_dims(preparar_imagem(p.read_bytes()), 0) for p in caminhos[idx]]\n    if modo == "multimodal":\n        amostra.append(dados["triagem"][idx:idx + 1])\n    np.testing.assert_allclose(recarregado(amostra, training=False).numpy(), previsoes[:1], atol=1e-5, rtol=1e-4)\n    for nome in [artefato.name, "geovision_model_pronto.json", "avaliacao.json", "historico.json"]:\n        shutil.copy2(local / nome, destino / nome)\n    print(f"Concluído em {(time.monotonic() - inicio) / 60:.1f} min. Artefatos experimentais: {destino}")\n    print("Avalie recall crítico, erros crítico->baixo e tamanho do teste com o revisor antes de homologar.")\n    return destino\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--raiz", default="/content/drive/MyDrive/GeoVision-IA")\n    parser.add_argument("--modo", choices=["visual", "multimodal"], default="multimodal")\n    parser.add_argument("--epocas", type=int, default=1_000_000)\n    parser.add_argument("--horas", type=float, default=3.0)\n    parser.add_argument("--avaliacao-exploratoria", action="store_true")\n    args = parser.parse_args()\n    treinar(args.raiz, modo=args.modo, epocas=args.epocas, horas=args.horas, avaliacao_exploratoria=args.avaliacao_exploratoria)\n', 'preparar_manifesto.py': '"""Inventário de imagens sem inventar identidade de edifícios.\n\nSão aceitas três estruturas, nesta ordem: ``01_imagens_brutas/<classe>``, o\ndataset já separado em ``02_dataset_limpo/<split>/<classe>`` e uma pasta que\ncontenha as quatro classes diretamente. A última forma é a recebida quando a\npasta compartilhada do Drive é adicionada ao Meu Drive como atalho.\n"""\nimport csv\nfrom pathlib import Path\n\nCLASSES = ("baixo", "critico", "medio", "sem_risco")\nEXTENSOES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}\n\n\ndef preparar_manifesto(raiz, exploratoria=False):\n    raiz = Path(raiz).resolve()\n    final = raiz / "imagens.csv"\n    if final.exists():\n        return final\n    rascunho = raiz / "imagens.preencher.csv"\n    if rascunho.exists() and not exploratoria:\n        return rascunho\n    destino = final if exploratoria else rascunho\n\n    def inventariar(pastas):\n        linhas = []\n        for pasta in pastas:\n            if not pasta.is_dir():\n                continue\n            for classe in sorted(pasta.iterdir()):\n                if not classe.is_dir():\n                    continue\n                fotos = sorted(p for p in classe.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSOES)\n                if fotos and classe.name not in CLASSES:\n                    raise ValueError(f"Pasta de classe não reconhecida: {classe.name}. Esperado: {CLASSES}")\n                for foto in fotos:\n                    if not foto.resolve().is_relative_to(raiz):\n                        raise ValueError(f"Imagem fora da raiz: {foto}")\n                    linhas.append({"caso_id": f"foto_{len(linhas)+1:06d}", "edificio_id": "",\n                                   "rotulo": classe.name, "imagem": foto.relative_to(raiz).as_posix()})\n        return linhas\n\n    # Não juntar fontes diferentes: cópias do mesmo arquivo inflariam o\n    # inventário, mesmo que a validação posterior elimine hashes repetidos.\n    linhas = inventariar([raiz / "01_imagens_brutas"])\n    if not linhas:\n        linhas = inventariar([raiz / "02_dataset_limpo" / split for split in ("train", "val", "test")])\n    if not linhas:\n        # A pasta compartilhada pode ser a própria raiz do dataset, com as\n        # quatro classes logo abaixo. Isso evita obrigar a recriar/mover o\n        # acervo apenas para atender ao layout de uma versão antiga.\n        linhas = inventariar([raiz])\n    if not linhas:\n        raise ValueError(\n            "Nenhuma imagem encontrada em 01_imagens_brutas/<classe>, "\n            "02_dataset_limpo/train|val|test/<classe> ou RAIZ/<classe>. "\n            "Confira RAIZ e adicione a pasta compartilhada ao Meu Drive."\n        )\n    with destino.open("x", encoding="utf-8-sig", newline="") as arquivo:\n        writer = csv.DictWriter(arquivo, fieldnames=["caso_id", "edificio_id", "rotulo", "imagem"])\n        writer.writeheader()\n        writer.writerows(linhas)\n    return destino\n\n\ndef conferir_manifesto(raiz, exploratoria=False):\n    """Verificação curta antes da GPU; a validação completa permanece no treino."""\n    arquivo = preparar_manifesto(raiz, exploratoria=exploratoria)\n    if arquivo.name != "imagens.csv":\n        raise ValueError(f"Inventário pronto: {arquivo}. Preencha edificio_id com o mesmo código para fotos do mesmo imóvel e salve como imagens.csv na mesma pasta. Não use um código diferente para cada foto sem conhecer a origem.")\n    with arquivo.open(encoding="utf-8-sig", newline="") as origem:\n        leitor = csv.DictReader(origem)\n        if not {"caso_id", "edificio_id", "rotulo", "imagem"}.issubset(leitor.fieldnames or []):\n            raise ValueError("imagens.csv deve ter caso_id,edificio_id,rotulo,imagem, separados por vírgula.")\n        linhas = list(leitor)\n    if not linhas:\n        raise ValueError("imagens.csv está vazio.")\n    faltantes = [i + 2 for i, linha in enumerate(linhas) if not (linha.get("edificio_id") or "").strip()]\n    if faltantes and not exploratoria:\n        raise ValueError(f"Preencha edificio_id em imagens.csv. Linhas pendentes: {faltantes[:10]}")\n    return arquivo\n', 'calculos_observacionais.py': '"""Medições auxiliares, sem limiares de ruína ou declaração de estabilidade.\n\nSó utilizar medições obtidas com segurança, instrumento e referência adequados.\nOs números não substituem análise estrutural, ensaios ou vistoria.\n"""\nimport math\n\n\ndef numero(valor, nome, minimo=0, estrito=False):\n    if isinstance(valor, bool):\n        raise ValueError(f"{nome}: booleano não é uma medição")\n    valor = float(valor)\n    if not math.isfinite(valor) or valor < minimo or (estrito and valor == minimo):\n        raise ValueError(f"{nome}: valor finito {\'maior que\' if estrito else \'>=\'} {minimo} necessário")\n    return valor\n\n\ndef abertura_mm(abertura_px, referencia_px, referencia_mm, *, mesmo_plano=False):\n    """Conversão local: exige escala coplanar e correção de perspectiva prévia."""\n    if mesmo_plano is not True:\n        raise ValueError("Sem escala no mesmo plano, não converter pixels para mm")\n    return numero(abertura_px, "abertura_px") * numero(referencia_mm, "referencia_mm", estrito=True) / numero(referencia_px, "referencia_px", estrito=True)\n\n\ndef evolucao_mm_dia(inicial_mm, final_mm, intervalo_horas):\n    """Mesma fissura e ponto; valor negativo pode ser fechamento/erro térmico."""\n    return (numero(final_mm, "final_mm") - numero(inicial_mm, "inicial_mm")) * 24 / numero(intervalo_horas, "intervalo_horas", estrito=True)\n\n\ndef desaprumo_mm_m(deslocamento_mm, altura_m):\n    return numero(deslocamento_mm, "deslocamento_mm") / numero(altura_m, "altura_m", estrito=True)\n\n\ndef distorcao_angular(recalque_diferencial_mm, distancia_m):\n    """Razão adimensional; não é limite admissível normativo."""\n    return numero(recalque_diferencial_mm, "recalque_diferencial_mm") / (1000 * numero(distancia_m, "distancia_m", estrito=True))\n\n\ndef gut_tecnico(gravidade, urgencia, tendencia):\n    """Tabela 25–27 do Manual, PDF pp.150–151. Exige notas técnicas explícitas.\n\n    Ausência retorna pendência; nunca preenche a lacuna com nota neutra.\n    Não converter softmax em gravidade nem usar o produto como probabilidade.\n    """\n    notas = (gravidade, urgencia, tendencia)\n    if any(n is None for n in notas):\n        return {"pontuacao": None, "estado": "incompleto"}\n    if any(isinstance(n, bool) or n not in (1, 3, 6, 8, 10) for n in notas):\n        raise ValueError("Notas técnicas devem pertencer a 1, 3, 6, 8, 10")\n    return {"pontuacao": math.prod(notas), "estado": "calculado"}\n'}
for nome, fonte in FONTES.items():
    Path('/content', nome).write_text(fonte, encoding='utf-8')
sys.path.insert(0, '/content')
import importlib
importlib.invalidate_caches()
for nome in FONTES:
    sys.modules.pop(Path(nome).stem, None)


## Configuração e dados

`imagens.csv`: caso_id, edificio_id, rotulo, imagem.

`casos.csv`: caso_id, edificio_id, rotulo, geral, detalhe, escala, revisor e
todos os campos categóricos abaixo. Medições são opcionais e só devem ser
preenchidas quando verificadas. Caminhos relativos à pasta GeoVision-IA.

Classes: baixo, critico, medio, sem_risco. `sem_risco` não atesta segurança.
Para imagens sem origem identificável, use avaliação exploratória: pelo menos cinco
arquivos de conteúdo diferente por classe. Ela não demonstra desempenho em novos imóveis.
Quando houver identificação, desative a opção exploratória e use pelo menos cinco
edifícios por classe. Nenhum desses mínimos garante suficiência estatística.


In [ ]:
from contrato_modelo import VOCABULARIO, MEDICOES
# No Drive, use “Organizar > Adicionar atalho ao Drive” na pasta compartilhada
# e selecione Meu Drive. Informe abaixo a pasta que contém baixo/critico/medio/sem_risco.
RAIZ = '/content/drive/MyDrive/GeoVision-IA'
MODO = 'visual'  # altere para 'multimodal' se já houver casos.csv revisado
AVALIACAO_EXPLORATORIA = True  # fotos sem identificação do imóvel; somente modo visual
HORAS = 3.0
MAX_EPOCAS = 1_000_000  # teto; quem controla a parada é o relógio
print('Categorias aceitas:', VOCABULARIO)
print('Medições opcionais e escalas numéricas (não limites):', MEDICOES)
if not Path(RAIZ).is_dir():
    raise FileNotFoundError(
        f'RAIZ não encontrada: {RAIZ}. Adicione a pasta compartilhada ao Meu Drive '
        'e informe aqui o caminho dela no Colab.'
    )


## Preparar a lista de imagens

Com `AVALIACAO_EXPLORATORIA=True`, cria `imagens.csv` automaticamente, com origem
do edifício em branco. Arquivos idênticos são deduplicados antes da separação.
Fotos diferentes do mesmo imóvel ainda podem vazar entre os conjuntos.
Com a opção desligada, cria `imagens.preencher.csv` para preencher os edifícios.
Usa `01_imagens_brutas/<classe>`; se não houver fotos nessa pasta, recupera as imagens
de `02_dataset_limpo/train|val|test/<classe>` ou de `RAIZ/<classe>`, que é a estrutura
da pasta compartilhada atual. Não mistura as fontes, não apaga arquivos e não extrai ZIPs.

Abra o CSV, preencha `edificio_id` com um identificador real de agrupamento (por exemplo,
`imovel_001` em todas as fotos do mesmo imóvel) e salve como `imagens.csv`, separado por
vírgulas, na mesma pasta. Não é necessário escrever endereços. O ID da foto não identifica
o edifício. Só preencha esse dado quando a origem for conhecida.


In [ ]:
from preparar_manifesto import preparar_manifesto
if MODO == 'visual':
    arquivo = preparar_manifesto(RAIZ, exploratoria=AVALIACAO_EXPLORATORIA)
    print('Arquivo:', arquivo)
    if arquivo.name != 'imagens.csv':
        print('Preencha edificio_id e salve como imagens.csv antes de treinar.')
else:
    print('Modo multimodal: prepare casos.csv com fotos, triagem e revisor dos casos reais.')


In [ ]:
from preparar_manifesto import conferir_manifesto
if MODO == 'visual':
    conferir_manifesto(RAIZ, exploratoria=AVALIACAO_EXPLORATORIA)
if AVALIACAO_EXPLORATORIA and MODO != 'visual':
    raise ValueError('Desative AVALIACAO_EXPLORATORIA para treinar casos multimodais revisados.')
from treinar_colab import treinar
destino = treinar(RAIZ, modo=MODO, horas=HORAS, epocas=MAX_EPOCAS, batch_imagens=8,
                 avaliacao_exploratoria=AVALIACAO_EXPLORATORIA)


In [ ]:
import json
print(json.dumps(json.loads((destino / 'avaliacao.json').read_text()), indent=2, ensure_ascii=False))
print('Modelo experimental salvo em:', destino)


## Cálculos observacionais: exemplo aritmético

Não são classificação de risco nem verificação resistente. Resultados verificados podem
preencher as colunas opcionais do CSV em uma execução futura. A escala da foto deve estar
no mesmo plano, com perspectiva corrigida; nunca se aproximar de risco para medir.


In [ ]:
from calculos_observacionais import evolucao_mm_dia, desaprumo_mm_m, distorcao_angular
print('Exemplo de evolução (mm/dia):', evolucao_mm_dia(0.4, 0.7, 48))
print('Exemplo de desaprumo (mm/m):', desaprumo_mm_m(6, 3))
print('Exemplo de distorção (adimensional):', distorcao_angular(5, 5))
